# 05 - Mean-Field Variational Inference

So far, variational inference has used a simple approximation $q(	heta)$.
In larger models, the latent variable is often multidimensional, for example
$z = (z_1, z_2, \dots, z_d)$.

A common simplification is the **mean-field assumption**:
$$
q(z) = \prod_{j=1}^d q_j(z_j)
$$
which assumes the variational distribution factorizes across coordinates.

This notebook covers:
1. What mean-field VI assumes
2. Why it makes optimization easier
3. What it cannot represent
4. A 2D Gaussian example

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## Part 1: Why factorize?

Suppose the true posterior is $p(z_1, z_2 \mid x)$.
If we allow any possible joint distribution, optimization becomes hard.
Mean-field VI restricts the approximation to the form
$$
q(z_1, z_2) = q_1(z_1) q_2(z_2)
$$
which removes variational dependence between the coordinates.

This is a big simplification:
- fewer parameters
- easier optimization
- often faster inference

But there is a cost: correlations in the true posterior may be lost.

In [ ]:
# A correlated 2D Gaussian will play the role of the true posterior
true_mean = np.array([1.0, -0.5])
true_cov = np.array([[1.0, 0.85], [0.85, 1.4]])
true_posterior = stats.multivariate_normal(mean=true_mean, cov=true_cov)

x = np.linspace(-3, 5, 200)
y = np.linspace(-4, 3, 200)
X, Y = np.meshgrid(x, y)
pos = np.dstack((X, Y))
Z_true = true_posterior.pdf(pos)

plt.figure(figsize=(6, 5))
plt.contour(X, Y, Z_true, levels=10, cmap='Blues')
plt.contourf(X, Y, Z_true, levels=10, cmap='Blues', alpha=0.5)
plt.xlabel('z1')
plt.ylabel('z2')
plt.title('True posterior: correlated 2D Gaussian')
plt.show()

## Part 2: A mean-field Gaussian family

A simple mean-field family is
$$
q(z_1, z_2) = athcal{N}(z_1; u_1, igma_1^2) athcal{N}(z_2; u_2, igma_2^2)
$$

This means the covariance matrix of $q$ is diagonal:
$$
igma_q = egin{bmatrix}
igma_1^2 & 0 \
0 & igma_2^2
nd{bmatrix}
$$
so $q$ cannot represent posterior correlation.

In [ ]:
def mean_field_pdf(grid_pos, mu1, mu2, sigma1, sigma2):
    z1 = grid_pos[..., 0]
    z2 = grid_pos[..., 1]
    return stats.norm.pdf(z1, mu1, sigma1) * stats.norm.pdf(z2, mu2, sigma2)

example_q = mean_field_pdf(pos, mu1=1.0, mu2=-0.5, sigma1=1.0, sigma2=1.2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
axes[0].contour(X, Y, Z_true, levels=10, cmap='Blues')
axes[0].contourf(X, Y, Z_true, levels=10, cmap='Blues', alpha=0.5)
axes[0].set_title('True correlated posterior')
axes[0].set_xlabel('z1')
axes[0].set_ylabel('z2')

axes[1].contour(X, Y, example_q, levels=10, cmap='Oranges')
axes[1].contourf(X, Y, example_q, levels=10, cmap='Oranges', alpha=0.5)
axes[1].set_title('Mean-field approximation')
axes[1].set_xlabel('z1')

plt.tight_layout()
plt.show()

print('Notice: the mean-field contours stay axis-aligned.')
print('They cannot tilt to capture correlation between z1 and z2.')

## Part 3: Optimizing the mean-field approximation

We now fit the best diagonal Gaussian $q$ to the true posterior by minimizing
$$
athrm{KL}(q(z_1, z_2) \| p(z_1, z_2 \mid x))
$$
using Monte Carlo estimates.

We parameterize $q$ by
$$
(u_1, u_2, og igma_1, og igma_2).
$$

In [ ]:
fixed_eps = np.random.normal(size=(6000, 2))

def sample_mean_field(params, eps):
    mu1, mu2, log_sigma1, log_sigma2 = params
    sigma1 = np.exp(log_sigma1)
    sigma2 = np.exp(log_sigma2)
    z = np.empty_like(eps)
    z[:, 0] = mu1 + sigma1 * eps[:, 0]
    z[:, 1] = mu2 + sigma2 * eps[:, 1]
    return z, sigma1, sigma2

def log_q_mean_field(z, params):
    mu1, mu2, log_sigma1, log_sigma2 = params
    sigma1 = np.exp(log_sigma1)
    sigma2 = np.exp(log_sigma2)
    return (
        stats.norm.logpdf(z[:, 0], mu1, sigma1) +
        stats.norm.logpdf(z[:, 1], mu2, sigma2)
    )

def log_p_true(z):
    return true_posterior.logpdf(z)

def estimate_elbo(params):
    z, _, _ = sample_mean_field(params, fixed_eps)
    return np.mean(log_p_true(z) - log_q_mean_field(z, params))

def objective(params):
    return -estimate_elbo(params)

In [ ]:
initial_params = np.array([0.0, 0.0, np.log(1.5), np.log(1.5)])
result = minimize(objective, initial_params, method='Nelder-Mead', options={'maxiter': 400, 'xatol': 1e-3, 'fatol': 1e-3})

mu1_opt, mu2_opt, log_sigma1_opt, log_sigma2_opt = result.x
sigma1_opt = np.exp(log_sigma1_opt)
sigma2_opt = np.exp(log_sigma2_opt)

print('Optimization success:', result.success)
print(f'Optimal mean: ({mu1_opt:.3f}, {mu2_opt:.3f})')
print(f'Optimal stds: ({sigma1_opt:.3f}, {sigma2_opt:.3f})')
print(f'Estimated ELBO: {-result.fun:.4f}')

In [ ]:
Z_q_opt = mean_field_pdf(pos, mu1_opt, mu2_opt, sigma1_opt, sigma2_opt)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)

axes[0].contour(X, Y, Z_true, levels=10, cmap='Blues')
axes[0].contourf(X, Y, Z_true, levels=10, cmap='Blues', alpha=0.5)
axes[0].set_title('True posterior')
axes[0].set_xlabel('z1')
axes[0].set_ylabel('z2')

axes[1].contour(X, Y, Z_q_opt, levels=10, cmap='Oranges')
axes[1].contourf(X, Y, Z_q_opt, levels=10, cmap='Oranges', alpha=0.5)
axes[1].set_title('Optimized mean-field q')
axes[1].set_xlabel('z1')

axes[2].contour(X, Y, Z_true, levels=10, colors='navy')
axes[2].contour(X, Y, Z_q_opt, levels=10, colors='darkorange')
axes[2].set_title('Overlay: p vs q')
axes[2].set_xlabel('z1')

plt.tight_layout()
plt.show()

## Part 4: What mean-field gets right and wrong

Mean-field VI often captures:
- approximate location of the posterior
- rough marginal uncertainty in each coordinate

But it often misses:
- posterior correlation
- curved or multimodal structure
- important joint dependencies

This is why mean-field VI can underestimate uncertainty in structured models.

In [ ]:
# Compare marginals of the true posterior and mean-field q
z1_grid = np.linspace(-3, 5, 500)
z2_grid = np.linspace(-4, 3, 500)

true_z1_marginal = stats.norm.pdf(z1_grid, loc=true_mean[0], scale=np.sqrt(true_cov[0, 0]))
true_z2_marginal = stats.norm.pdf(z2_grid, loc=true_mean[1], scale=np.sqrt(true_cov[1, 1]))
q_z1_marginal = stats.norm.pdf(z1_grid, loc=mu1_opt, scale=sigma1_opt)
q_z2_marginal = stats.norm.pdf(z2_grid, loc=mu2_opt, scale=sigma2_opt)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(z1_grid, true_z1_marginal, label='true marginal', linewidth=3, color='navy')
axes[0].plot(z1_grid, q_z1_marginal, label='mean-field marginal', linewidth=2, color='darkorange')
axes[0].set_title('Marginal for z1')
axes[0].set_xlabel('z1')
axes[0].legend()

axes[1].plot(z2_grid, true_z2_marginal, label='true marginal', linewidth=3, color='navy')
axes[1].plot(z2_grid, q_z2_marginal, label='mean-field marginal', linewidth=2, color='darkorange')
axes[1].set_title('Marginal for z2')
axes[1].set_xlabel('z2')
axes[1].legend()

plt.tight_layout()
plt.show()

## Summary

What to remember:
1. Mean-field VI assumes the variational distribution factorizes
2. This makes optimization much easier
3. The approximation often captures marginals reasonably well
4. But it cannot represent posterior correlations

Mean-field VI is simple and fast, which is why it is used so often, but its independence assumption is also its main limitation.

In [ ]:
# Exercises
# 1) Change true_cov to increase or decrease the correlation. How does the fit change?
# 2) Set the off-diagonal covariance to 0. What happens then?
# 3) Move the true mean and re-run the optimization.

pass